# Example 22 — Inverse conduction: measuring thermal diffusivity

The practical one. You record **40 noisy 'thermocouple' readings** scattered in space and
time on the Example 15 slab; the diffusivity $\alpha$ of the material is unknown:
$$u_t = \alpha\,u_{xx},\qquad \alpha = ?\qquad \text{(no IC, no BC given to the solver)}$$

Exactly Example 2's recipe: $\alpha$ becomes `nn.Parameter` (through a log-transform for
positivity — Example 3's trick), and the physics + sparse data pin it down jointly.
**This is how flash-method diffusivity measurement works** — the PINN version is 5 lines.

Verified: recovered $\alpha$ = 0.1942 vs true 0.2 (from a wrong initial
guess of 0.05, with 2% noise), ~46 s on CPU.

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

def g1(f, x):
    return torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0]

A_TRUE, T = 0.2, 0.5
def u_true(x, t):   # 'nature' — used only to fake the thermocouple data
    return np.exp(-A_TRUE*np.pi**2*t)*np.sin(np.pi*x) + 0.3*np.exp(-9*A_TRUE*np.pi**2*t)*np.sin(3*np.pi*x)

xd = np.random.rand(40); td = np.random.rand(40)*T
ud = u_true(xd, td) + 0.02*np.random.randn(40)
xd = torch.tensor(xd, dtype=torch.float32, device=device).reshape(-1,1)
td = torch.tensor(td, dtype=torch.float32, device=device).reshape(-1,1)
ud = torch.tensor(ud, dtype=torch.float32, device=device).reshape(-1,1)

net = nn.Sequential(nn.Linear(2,48), nn.Tanh(), nn.Linear(48,48), nn.Tanh(),
                    nn.Linear(48,48), nn.Tanh(), nn.Linear(48,1)).to(device)
log_a = nn.Parameter(torch.log(torch.tensor(0.05, device=device)))   # wrong guess: 0.05
opt = torch.optim.Adam(list(net.parameters()) + [log_a], 2e-3)

hist = []
t0 = time.perf_counter()
for e in range(9000):
    if e == 6000:
        for g in opt.param_groups: g['lr'] = 4e-4
    opt.zero_grad()
    a = torch.exp(log_a)
    x = torch.rand(2000,1,device=device).requires_grad_(True)
    t = (torch.rand(2000,1,device=device)*T).requires_grad_(True)
    u = net(torch.cat([x,t],1))
    res = g1(u,t) - a*g1(g1(u,x),x)
    loss = (res**2).mean() + 30*((net(torch.cat([xd,td],1)) - ud)**2).mean()
    loss.backward(); opt.step()
    hist.append(float(torch.exp(log_a)))
if device.type=='cuda': torch.cuda.synchronize()
print(f'training: {time.perf_counter()-t0:.0f} s')
print(f'recovered alpha = {hist[-1]:.4f}   (true {A_TRUE})')

fig, ax = plt.subplots(1, 2, figsize=(11.5,4))
ax[0].plot(hist, 'b'); ax[0].axhline(A_TRUE, color='g', ls='--', label=f'true α = {A_TRUE}')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('learned α'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[0].set_title(f'Diffusivity recovery: {hist[-1]:.4f}')
xg = torch.linspace(0,1,200,device=device).reshape(-1,1)
for tv, c in zip((0.05,0.2,0.45), ('tab:red','tab:green','tab:blue')):
    with torch.no_grad(): up = net(torch.cat([xg, torch.full_like(xg,tv)],1)).cpu().numpy().ravel()
    ax[1].plot(xg.cpu(), u_true(xg.cpu().numpy().ravel(), tv), c, lw=2, alpha=.5)
    ax[1].plot(xg.cpu(), up, '--', color=c, lw=1.3, label=f't={tv}')
mask = td.cpu().numpy().ravel() < 0.5
ax[1].scatter(xd.cpu().numpy().ravel(), ud.cpu().numpy().ravel(), s=14, c='k', label='noisy data')
ax[1].set_xlabel('x'); ax[1].set_ylabel('u'); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
ax[1].set_title('Reconstructed field (solid=true, dashed=PINN)')
plt.tight_layout(); plt.show()

## Observations
- **A property measurement, not a curve fit:** the network reconstructs the whole field AND
  the material constant from scattered noisy points — no IC/BC supplied. This is the
  flash-method / thermal-metrology use case, and the single strongest 'sell' of PINNs to an
  experimentalist.
- **Positivity via log-α** (Example 3): negative diffusivity is a backwards heat equation —
  ill-posed — so we make it unreachable.
- **Sensitivity intuition:** early-time data carries most of the α information (mode-3
  content dies by t≈0.1 — Example 15); try restricting data to t>0.3 and watch the
  recovery degrade.

**Try:** fewer sensors (10? 5?); more noise; recover α *and* an unknown heat source
simultaneously (two-parameter inverse, Example 3's pattern).